# Notebook 06: Stage 2 Evaluation & Production Model Training
This notebook executes a strict, **leak-free 5-fold cross-validation** across three backbone architectures (`EfficientNet-B0`, `ConvNeXt-Nano`, and `Swin-Tiny`) using tuned Stage 2 attention parameters. 

Following cross-validation, production-ready Stage 2 attention heads are trained using backbones trained on the full dataset and saved alongside calibrated decision thresholds.

## 1. Setup & Environment
Imports essential dependencies, sets random seeds for reproducibility, configures local/Kaggle dataset paths and verifies GPU availability.

In [1]:
import os, json, time, gc
import numpy as np
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from sklearn.metrics import roc_curve

import sys
sys.path.append('/kaggle/input/datasets/mfjmrizvi/cbis-ddsm-project-config')
from abmil_common import (
    build_backbone, PatchClassifier, BagClassifier, CachedBagDataset,
    collate_cached, extract_features, compute_all_metrics,
    get_normalisation_tensors)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

Device: cuda


In [2]:
NB02 = Path("/kaggle/input/notebooks/mfjmrizvi/02-mil-patch-extraction")
NB03 = Path("/kaggle/input/notebooks/mfjmrizvi/03-efficientnet-b0")
NB04 = Path("/kaggle/input/notebooks/mfjmrizvi/04-convnext-nano")
NB05 = Path("/kaggle/input/notebooks/mfjmrizvi/05-swint")
NB027 = Path("/kaggle/input/notebooks/mfjmrizvi/02-7-stage-2-head-search") 
OUT  = Path("/kaggle/working")

SEED = 42
PATCH_SIZE = 224
EPOCHS_S2 = 30
PATIENCE_S2 = 7

torch.manual_seed(SEED)
np.random.seed(SEED)

_mean_gpu, _std_gpu = get_normalisation_tensors(DEVICE)

In [3]:
# Load NB02 outputs
X_train_all  = np.load(NB02 / "X_train_patches.npy")
y_train_all  = np.load(NB02 / "y_train_labels.npy")
bag_ids_all  = np.load(NB02 / "bag_ids_train.npy")
fold_ids     = np.load(NB02 / "fold_ids.npy")
all_bags     = np.unique(bag_ids_all)

print(f"Loaded: {X_train_all.shape} patches, {len(all_bags)} bags")

Loaded: (58820, 224, 224, 1) patches, 1226 bags


## 2. Model & Pipeline Configurations

Defining paths for each architecture’s Stage 1 checkpoints (`fold0`–`fold4` & `full`) and Optuna-optimised Stage 2 hyperparameter configurations.

### 2.1 Model & Hyperparameter Configurations
Defines the paths for Stage 1 backbone weights (`NB03`, `NB04`, `NB05`) alongside the Optuna-tuned Stage 2 hyperparameter paths for `EfficientNet-B0`, `ConvNeXt-Nano`, and `Swin-Tiny`.

In [4]:
architectures = [
    {
        "model_name": "efficientnet_b0",
        "tag": "effnet_b0",
        "s1_dir": NB03,
        "s2_parameter_path": NB027 / "effnet_b0_stage2_optuna_study.json",
    },
    {
        "model_name": "convnext_nano",
        "tag": "convnext_nano",
        "s1_dir": NB04,
        "s2_parameter_path": NB027 / "convnext_nano_stage2_optuna_study.json",
    },
    {
        "model_name": "swin_tiny_patch4_window7_224",
        "tag": "swin_t",
        "s1_dir": NB05,
        "s2_parameter_path": NB027 / "swin_t_stage2_optuna_study.json",
    },
]

### 2.2 Stage 1 Feature Extraction Pipeline
Loads a trained Stage 1 backbone, freezes its parameters to prevent gradient updates and extracts low-dimensional patch feature representations for all input patches (`X_all`).

In [5]:
def load_backbone_features(model_name, stage1_ckpt_path, X_all, device, patch_size=224):
    # Load a Stage 1 checkpoint and extract features for the given patches.
    backbone = build_backbone(model_name, pretrained=False).to(device)
    with torch.no_grad():
        _dummy = torch.zeros(2, 3, patch_size, patch_size, device=device)
        feat_dim = backbone(_dummy).shape[1]
    del _dummy

    patch_model = PatchClassifier(backbone, feat_dim).to(device)
    patch_model.load_state_dict(torch.load(stage1_ckpt_path, map_location=device))
    feature_extractor = patch_model.backbone
    feature_extractor.eval()
    for p in feature_extractor.parameters():
        p.requires_grad_(False)

    feats = extract_features(X_all, feature_extractor, _mean_gpu, _std_gpu, device)

    del backbone, patch_model, feature_extractor
    gc.collect(); torch.cuda.empty_cache()
    return feats, feat_dim

### 2.3 Stage 2 MIL Head Training & Evaluation
Initialises a gated Multiple Instance Learning (`BagClassifier`) head with tuned hyperparameters (`attn_dim`, `dropout`, `lr`, `weight_decay`). Trains on designated bag splits using early stopping on validation loss, returning the best-performing model state alongside validation probabilities.

In [ ]:
def train_stage2_head(feats_all, y_all, bag_ids_all_, feat_dim,
                       train_bags, val_bags, parameter, device,
                       epochs=30, patience=7):
    # Trains ONE Stage 2 attention head on given train/val bags,
    # using a fixed parameter (attn_dim/dropout/lr/weight_decay).
    attn_dim     = parameter["attn_dim"]
    dropout      = parameter["dropout"]
    lr           = parameter["lr"]
    weight_decay = parameter["weight_decay"]

    train_mask = np.isin(bag_ids_all_, train_bags)
    val_mask   = np.isin(bag_ids_all_, val_bags)

    train_ds = CachedBagDataset(feats_all[train_mask], y_all[train_mask],
                                 bag_ids_all_[train_mask], train_bags)
    val_ds   = CachedBagDataset(feats_all[val_mask], y_all[val_mask],
                                 bag_ids_all_[val_mask], val_bags)
    train_dl = DataLoader(train_ds, batch_size=1, shuffle=True, collate_fn=collate_cached)
    val_dl   = DataLoader(val_ds, batch_size=1, shuffle=False, collate_fn=collate_cached)

    model = BagClassifier(feat_dim, attn_dim, dropout=dropout, gated=True).to(device)
    criterion = nn.BCEWithLogitsLoss()
    optimiser = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimiser, mode='min', factor=0.5, patience=3)

    best_val_loss, patience_ctr, best_state = float("inf"), 0, None

    for epoch in range(1, epochs + 1):
        model.train()
        for h_list, labels in train_dl:
            h, label = h_list[0].to(device), labels[0].to(device)
            optimiser.zero_grad()
            logit, _ = model(h)
            loss = criterion(logit, label)
            loss.backward()
            optimiser.step()

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for h_list, labels in val_dl:
                h, label = h_list[0].to(device), labels[0].to(device)
                logit, _ = model(h)
                val_loss += criterion(logit, label).item()
        val_loss /= len(val_ds)
        scheduler.step(val_loss)

        if val_loss < best_val_loss:
            best_val_loss, patience_ctr = val_loss, 0
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            patience_ctr += 1
            if patience_ctr >= patience:
                break

    model.load_state_dict(best_state)
    model.eval()

    val_probs, val_true = [], []
    with torch.no_grad():
        for h_list, labels in val_dl:
            logit, _ = model(h_list[0].to(device))
            val_probs.append(torch.sigmoid(logit).item())
            val_true.append(labels[0].item())
    val_probs, val_true = np.array(val_probs), np.array(val_true)

    return model, val_probs, val_true, best_val_loss

## 3. Leak-Free 5-Fold Cross-Validation

To prevent data leakage:
1. **Fold $f$** uses a Stage 1 backbone trained strictly on the other 4 folds.
2. Features are extracted dynamically per fold.
3. A fresh Stage 2 attention head is trained on $4$ folds and evaluated on the held-out fold $f$.
4. Metrics (AUC, F1, Sensitivity, Specificity, FPR, MCC, ECE) are aggregated across all 5 folds per architecture.

In [6]:
# Leak-free 5-fold CV
all_cv_results = {}

for arch in architectures:
    model_name = arch["model_name"]
    tag = arch["tag"]
    print(f"\n{'='*70}\nStep 4, Real 5-fold CV: {tag}\n{'='*70}")

    with open(arch["s2_parameter_path"]) as f:
        parameter = json.load(f)["best_params"]
    print(f"Using Stage 2 parameter: {parameter}")

    fold_results = []

    for fold in range(5):
        print(f"\n  --- Fold {fold} ---")
        s1_ckpt = arch["s1_dir"] / f"{model_name}_stage1_fold{fold}.pth"
        print(f"  Using backbone: {s1_ckpt.name}")

        feats_all, feat_dim = load_backbone_features(
            model_name, s1_ckpt, X_train_all, DEVICE
        )

        train_bags = all_bags[fold_ids[all_bags] != fold]
        val_bags   = all_bags[fold_ids[all_bags] == fold]

        _, val_probs, val_true, best_val_loss = train_stage2_head(
            feats_all, y_train_all, bag_ids_all, feat_dim,
            train_bags, val_bags, parameter, DEVICE,
            epochs=EPOCHS_S2, patience=PATIENCE_S2
        )

        fpr_c, tpr_c, thresholds_c = roc_curve(val_true, val_probs)
        idx = np.argmax(tpr_c >= 0.90)
        thresh = float(thresholds_c[idx])

        m = compute_all_metrics(val_true, val_probs, thresh, f"[{tag}] Fold {fold}")
        m["fold"] = fold
        fold_results.append(m)

        del feats_all
        gc.collect(); torch.cuda.empty_cache()

    # Aggregate
    import pandas as pd
    cv_df = pd.DataFrame(fold_results)
    summary = cv_df[["auc","f1","sensitivity","specificity","fpr","mcc","ece"]].agg(["mean","std"])
    print(f"\n{tag} 5-fold CV summary:")
    print(summary)

    all_cv_results[tag] = {
        "model": model_name,
        "stage1_source": "per-fold dedicated backbones (leak-free)",
        "stage2_parameter": parameter,
        "fold_results": fold_results,
        "cv_mean": summary.loc["mean"].to_dict(),
        "cv_std": summary.loc["std"].to_dict(),
    }

    with open(OUT / f"{tag}_real_cv_results.json", "w") as f:
        json.dump(all_cv_results[tag], f, indent=2)

print("\nComplete: leak-free CV done for all three architectures.")


Step 4 — Real 5-fold CV: effnet_b0
Using Stage 2 parameter: {'attn_dim': 256, 'dropout': 0.1898021217101117, 'lr': 0.0008448646807339096, 'weight_decay': 0.0002949971987301991}

  --- Fold 0 ---
  Using backbone: efficientnet_b0_stage1_fold0.pth
── [effnet_b0] Fold 0 ──
  threshold      : 0.1701
  auc            : 0.8025
  f1             : 0.7235
  sensitivity    : 0.9060
  specificity    : 0.4355
  fpr            : 0.5645
  mcc            : 0.3845
  ece            : 0.0833
  confusion      : TN=54 FP=70 FN=11 TP=106

  --- Fold 1 ---
  Using backbone: efficientnet_b0_stage1_fold1.pth
── [effnet_b0] Fold 1 ──
  threshold      : 0.1268
  auc            : 0.6519
  f1             : 0.6601
  sensitivity    : 0.9174
  specificity    : 0.2769
  fpr            : 0.7231
  mcc            : 0.2476
  ece            : 0.1875
  confusion      : TN=36 FP=94 FN=9 TP=100

  --- Fold 2 ---
  Using backbone: efficientnet_b0_stage1_fold2.pth
── [effnet_b0] Fold 2 ──
  threshold      : 0.2452
  auc      

## 4. Production Stage 2 Training & Threshold Calibration

For each architecture:
1. **Full Feature Extraction**: Extracted using the Stage 1 backbone trained on the complete dataset (`stage1_full.pth`).
2. **Monitoring Split**: A 10% holdout split is reserved strictly for early stopping and threshold calibration (completely independent of sealed test data).
3. **Threshold Calibration**: A decision threshold is calibrated to guarantee **$\ge 90\%$ Sensitivity** for medical screening priorities.
4. **Export**: Production weights and calibration manifests are saved for final test set inference.

In [9]:
production_models = {}

for arch in architectures:
    model_name = arch["model_name"]
    tag = arch["tag"]
    print(f"\n{'='*70}\nStep 5 Production model: {tag}\n{'='*70}")

    with open(arch["s2_parameter_path"]) as f:
        parameter = json.load(f)["best_params"]

    s1_ckpt = arch["s1_dir"] / f"{model_name}_stage1_full.pth"
    print(f"Using backbone: {s1_ckpt.name}")

    feats_all, feat_dim = load_backbone_features(
        model_name, s1_ckpt, X_train_all, DEVICE
    )

    rng = np.random.default_rng(999)
    shuffled = rng.permutation(all_bags)
    n_val = int(0.1 * len(shuffled))
    val_bags   = shuffled[:n_val]
    train_bags = shuffled[n_val:]

    model, val_probs, val_true, best_val_loss = train_stage2_head(
        feats_all, y_train_all, bag_ids_all, feat_dim,
        train_bags, val_bags, parameter, DEVICE,
        epochs=EPOCHS_S2, patience=PATIENCE_S2
    )

    # Calibrate a threshold on this monitoring split
    # sensitivity>=0.90 policy used everywhere else in the pipeline 
    fpr_c, tpr_c, thresholds_c = roc_curve(val_true, val_probs)
    valid_idx = np.where(tpr_c >= 0.90)[0]
    if len(valid_idx) == 0:
        idx = np.argmax(tpr_c)   # fallback if 90% sensitivity unreachable
    else:
        idx = valid_idx[0]
    calibrated_threshold = float(thresholds_c[idx])
    threshold_sensitivity = float(tpr_c[idx])
    threshold_fpr = float(fpr_c[idx])

    print(f"  Calibrated threshold: {calibrated_threshold:.4f}  "
          f"(sensitivity={threshold_sensitivity:.4f}, fpr={threshold_fpr:.4f})")

    save_path = OUT / f"{model_name}_production_stage2.pth"
    torch.save(model.state_dict(), save_path)
    print(f"✓ Saved production Stage 2 head: {save_path.name}")

    production_models[tag] = {
        "model": model_name,
        "stage1_checkpoint": str(s1_ckpt),
        "stage2_checkpoint": str(save_path),
        "stage2_parameter": parameter,
        "internal_val_best_loss": best_val_loss,
        "calibrated_threshold": calibrated_threshold,
        "threshold_val_sensitivity": threshold_sensitivity,
        "threshold_val_fpr": threshold_fpr,
        "note": "Threshold calibrated on the internal 10% monitoring split (never one of the 5 CV folds, never the sealed test set).",
    }

    del feats_all, model
    gc.collect(); torch.cuda.empty_cache()

with open(OUT / "production_models_manifest.json", "w") as f:
    json.dump(production_models, f, indent=2)

print("\nStep 5 complete, production Stage 2 heads and calibrated thresholds saved for all three architectures.")


Step 5 Production model: effnet_b0
Using backbone: efficientnet_b0_stage1_full.pth
  Calibrated threshold: 0.3435  (sensitivity=0.9000, fpr=0.5806)
✓ Saved production Stage 2 head: efficientnet_b0_production_stage2.pth

Step 5 Production model: convnext_nano
Using backbone: convnext_nano_stage1_full.pth
  Calibrated threshold: 0.1229  (sensitivity=0.9167, fpr=0.6129)
✓ Saved production Stage 2 head: convnext_nano_production_stage2.pth

Step 5 Production model: swin_t
Using backbone: swin_tiny_patch4_window7_224_stage1_full.pth
  Calibrated threshold: 0.2210  (sensitivity=0.9000, fpr=0.6774)
✓ Saved production Stage 2 head: swin_tiny_patch4_window7_224_production_stage2.pth

Step 5 complete, production Stage 2 heads and calibrated thresholds saved for all three architectures.
